In [1]:
!nvidia-smi

Thu Oct  8 07:37:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:

from numba import cuda

device = cuda.get_current_device()

name = device.name
if isinstance(name, bytes):
    name = name.decode("utf-8")

print("Device Name:", name)
print("Compute Capability:", device.compute_capability)
print("Multiprocessor Count:", device.MULTIPROCESSOR_COUNT)
print("Max Threads Per Block:", device.MAX_THREADS_PER_BLOCK)
print("Max Block Dimensions:",
      device.MAX_BLOCK_DIM_X,
      device.MAX_BLOCK_DIM_Y,
      device.MAX_BLOCK_DIM_Z)
print("Warp Size:", device.WARP_SIZE)



Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count: 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024 1024 64
Warp Size: 32


In [4]:

import numpy as np
import time
from numba import cuda

# CPU Implementation
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

# GPU Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0


In [5]:

def benchmark_run(N, student_id_seed=230103054):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # CPU measurement
    if N <= 1_000_000:
        t0 = time.perf_counter()
        double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000
    else:
        cpu_time_ms = float('nan')

    # GPU configuration
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # GPU warmup
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # GPU total time
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)

    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()

    gpu_total_ms = (time.perf_counter() - t0) * 1000

    # GPU kernel-only time
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms


In [6]:

sizes = [100, 1000, 10000, 100000, 1000000, 5000000, 10000000]

results = []

print(f"{'N':<12} {'CPU (ms)':<15} {'GPU Total (ms)':<18} {'GPU Kernel (ms)':<18} {'Winner'}")
print("-" * 80)

for N in sizes:
    cpu, gpu_total, gpu_kernel = benchmark_run(N)

    if np.isnan(cpu):
        winner = "N/A"
    else:
        winner = "GPU" if gpu_total < cpu else "CPU"

    results.append((N, cpu, gpu_total, gpu_kernel, winner))

    print(f"{N:<12} {cpu:<15.4f} {gpu_total:<18.4f} {gpu_kernel:<18.4f} {winner}")


N            CPU (ms)        GPU Total (ms)     GPU Kernel (ms)    Winner
--------------------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


100          0.0599          1.3201             0.0820             CPU
1000         0.2480          0.5939             0.0569             CPU
10000        2.3740          0.5130             0.0676             GPU
100000       23.7266         0.6825             0.0631             GPU
1000000      376.4055        3.3998             0.1643             GPU
5000000      nan             14.0786            0.2813             N/A
10000000     nan             25.9173            0.4615             N/A


In [7]:

@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    d_arr[idx] = 999.0

N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = 4

print("Launching defective kernel with 1024 threads on 1000-element array...")

defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()

print("Execution finished. Checking host copy...")

h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")


Launching defective kernel with 1024 threads on 1000-element array...
Execution finished. Checking host copy...
Elements processed: 1000. Last element: 999.0


In [8]:

@cuda.jit
def inspect_threads_kernel(log_block, log_thread,
                           log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)

    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

# Configuration
N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

size = blocks_per_grid * threads_per_block

log_b = cuda.device_array(size, dtype=np.int32)
log_t = cuda.device_array(size, dtype=np.int32)
log_g = cuda.device_array(size, dtype=np.int32)
log_a = cuda.device_array(size, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](
    log_b, log_t, log_g, log_a, N
)

cuda.synchronize()

# Copy results from GPU to CPU
b = log_b.copy_to_host()
t = log_t.copy_to_host()
g = log_g.copy_to_host()
a = log_a.copy_to_host()

print("Global ID | Block ID | Thread ID | Status")
print("-" * 50)

for i in range(32, 40):
    status = "ACTIVE" if a[i] == 1 else "IDLE (GUARDED)"
    print(f"{g[i]:<9} | {b[i]:<8} | {t[i]:<9} | {status}")


Global ID | Block ID | Thread ID | Status
--------------------------------------------------
32        | 4        | 0         | ACTIVE
33        | 4        | 1         | ACTIVE
34        | 4        | 2         | ACTIVE
35        | 4        | 3         | ACTIVE
36        | 4        | 4         | ACTIVE
37        | 4        | 5         | IDLE (GUARDED)
38        | 4        | 6         | IDLE (GUARDED)
39        | 4        | 7         | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [9]:

@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])


In [10]:

N = 50000

h_arr = np.ones(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads = 256
blocks = (N + threads - 1) // threads

naive_times = []

print("RACE CONDITION EXPERIMENT")
print("-" * 60)

for trial in range(5):
    d_total = cuda.to_device(
        np.array([0.0], dtype=np.float32)
    )

    start = time.perf_counter()

    naive_sum_kernel[blocks, threads](d_arr, d_total)
    cuda.synchronize()

    elapsed = (time.perf_counter() - start) * 1000
    result = d_total.copy_to_host()[0]

    naive_times.append(elapsed)

    print(f"Trial {trial + 1}")
    print(f"Expected Sum: 50000.0")
    print(f"Naive Kernel Output: {result}")
    print(f"Lost Updates: {50000.0 - result}")
    print(f"Execution Time: {elapsed:.4f} ms")
    print("-" * 60)


RACE CONDITION EXPERIMENT
------------------------------------------------------------
Trial 1
Expected Sum: 50000.0
Naive Kernel Output: 2.0
Lost Updates: 49998.0
Execution Time: 68.3039 ms
------------------------------------------------------------
Trial 2
Expected Sum: 50000.0
Naive Kernel Output: 2.0
Lost Updates: 49998.0
Execution Time: 0.1848 ms
------------------------------------------------------------
Trial 3
Expected Sum: 50000.0
Naive Kernel Output: 2.0
Lost Updates: 49998.0
Execution Time: 0.1171 ms
------------------------------------------------------------
Trial 4
Expected Sum: 50000.0
Naive Kernel Output: 2.0
Lost Updates: 49998.0
Execution Time: 0.1098 ms
------------------------------------------------------------
Trial 5
Expected Sum: 50000.0
Naive Kernel Output: 2.0
Lost Updates: 49998.0
Execution Time: 0.1116 ms
------------------------------------------------------------


In [11]:

# Atomic Sum Experiment

d_total = cuda.to_device(
    np.array([0.0], dtype=np.float32)
)

# Warmup to exclude first-time compilation
atomic_sum_kernel[blocks, threads](d_arr, d_total)
cuda.synchronize()

# Reset sum
d_total.copy_to_device(
    np.array([0.0], dtype=np.float32)
)

# Measure atomic execution
start = time.perf_counter()

atomic_sum_kernel[blocks, threads](d_arr, d_total)
cuda.synchronize()

atomic_time = (time.perf_counter() - start) * 1000
atomic_result = d_total.copy_to_host()[0]

print("ATOMIC SUM EXPERIMENT")
print("-" * 50)
print(f"Expected Sum: 50000.0")
print(f"Atomic Sum Output: {atomic_result}")
print(f"Correct Result: {atomic_result == 50000.0}")
print(f"Atomic Kernel Time: {atomic_time:.4f} ms")
print(f"Average Naive Kernel Time: {np.mean(naive_times[1:]):.4f} ms")


ATOMIC SUM EXPERIMENT
--------------------------------------------------
Expected Sum: 50000.0
Atomic Sum Output: 50000.0
Correct Result: True
Atomic Kernel Time: 0.4575 ms
Average Naive Kernel Time: 0.1308 ms


In [12]:

import math

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)

    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)

        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))

        intensity = 1.0 - (dist / max_dist)

        if intensity < 0.0:
            intensity = 0.0

        d_canvas[y, x] = intensity

# 2D execution configuration
WIDTH, HEIGHT = 1024, 1024

canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

# 16 x 16 threads per block
threads_2d = (16, 16)

blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

radial_vignette_2d[blocks_2d, threads_2d](
    d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2
)

cuda.synchronize()

result_canvas = d_canvas.copy_to_host()

print("2D GRID EXPERIMENT")
print("-" * 50)
print("Blocks in X:", blocks_2d[0])
print("Blocks in Y:", blocks_2d[1])
print("Total Blocks:", blocks_2d[0] * blocks_2d[1])
print("Matrix Shape:", result_canvas.shape)
print("Center Pixel (512,512):", result_canvas[512, 512])
print("Corner Pixel (0,0):", result_canvas[0, 0])


2D GRID EXPERIMENT
--------------------------------------------------
Blocks in X: 64
Blocks in Y: 64
Total Blocks: 4096
Matrix Shape: (1024, 1024)
Center Pixel (512,512): 1.0
Corner Pixel (0,0): 0.0


In [14]:

import hashlib

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()

    h.update(student_id_str.strip().encode('utf-8'))
    h.update(cuda.get_current_device().name.encode('utf-8'))
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))

    digest = h.hexdigest()[:16].upper()

    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)

    return digest

STUDENT_ID = "230103054"
CROSSOVER_N = 10000

generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)


OFFICIAL VERIFICATION CHECKSUM TOKEN: 6BD72AB71A4D3AA7


'6BD72AB71A4D3AA7'